In [14]:
!pip install -q langchain langchain-groq langgraph ddgs pydantic

import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

In [15]:
import sqlite3

DB_PATH = "lab_booking.db"


def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_conn()
    cur = conn.cursor()

    cur.executescript("""
    CREATE TABLE IF NOT EXISTS tests (
        test_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        fasting_required INTEGER NOT NULL
    );

    CREATE TABLE IF NOT EXISTS slots (
        slot_id TEXT PRIMARY KEY,
        test_id TEXT NOT NULL,
        slot_time TEXT NOT NULL,
        is_booked INTEGER DEFAULT 0,
        FOREIGN KEY (test_id) REFERENCES tests(test_id)
    );

    CREATE TABLE IF NOT EXISTS bookings (
        booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT NOT NULL,
        test_id TEXT NOT NULL,
        slot_id TEXT NOT NULL,
        status TEXT DEFAULT 'CONFIRMED',
        FOREIGN KEY (test_id) REFERENCES tests(test_id),
        FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
    );
    """)

    # 8 tests across 3 categories (Blood, Imaging, Cardiac)
    tests_data = [
        ("T1", "Complete Blood Count (CBC)", "Blood", 400.0, 0),
        ("T2", "Lipid Profile", "Blood", 850.0, 1),
        ("T3", "HbA1c (Diabetes)", "Blood", 600.0, 0),
        ("T4", "Thyroid Profile (T3 T4 TSH)", "Blood", 750.0, 0),
        ("T5", "Chest X-Ray", "Imaging", 500.0, 0),
        ("T6", "Abdominal Ultrasound", "Imaging", 1200.0, 1),
        ("T7", "ECG (Electrocardiogram)", "Cardiac", 350.0, 0),
        ("T8", "2D Echo", "Cardiac", 1800.0, 0),
    ]
    cur.executemany(
        "INSERT OR IGNORE INTO tests VALUES (?, ?, ?, ?, ?)", tests_data
    )

    # 3+ slots per test; T8 has NO free slots (all is_booked = 1)
    slots_data = [
        ("S1", "T1", "2026-11-20 09:00", 0),
        ("S2", "T1", "2026-11-20 10:00", 0),
        ("S3", "T1", "2026-11-20 11:00", 1),
        ("S4", "T2", "2026-11-20 08:00", 0),
        ("S5", "T2", "2026-11-20 08:30", 0),
        ("S6", "T2", "2026-11-20 09:00", 0),
        ("S7", "T3", "2026-11-20 10:00", 0),
        ("S8", "T3", "2026-11-20 11:00", 0),
        ("S9", "T3", "2026-11-20 12:00", 0),
        ("S10", "T4", "2026-11-20 09:30", 0),
        ("S11", "T4", "2026-11-20 10:30", 0),
        ("S12", "T4", "2026-11-20 11:30", 0),
        ("S13", "T5", "2026-11-20 14:00", 0),
        ("S14", "T5", "2026-11-20 14:30", 0),
        ("S15", "T5", "2026-11-20 15:00", 0),
        ("S16", "T6", "2026-11-20 09:00", 0),
        ("S17", "T6", "2026-11-20 10:00", 0),
        ("S18", "T6", "2026-11-20 11:00", 0),
        ("S19", "T7", "2026-11-20 16:00", 0),
        ("S20", "T7", "2026-11-20 16:30", 0),
        ("S21", "T7", "2026-11-20 17:00", 0),
        # T8 (2D Echo) - all booked (no free slots)
        ("S22", "T8", "2026-11-20 10:00", 1),
        ("S23", "T8", "2026-11-20 11:00", 1),
        ("S24", "T8", "2026-11-20 12:00", 1),
    ]
    cur.executemany(
        "INSERT OR IGNORE INTO slots VALUES (?, ?, ?, ?)", slots_data
    )

    conn.commit()
    conn.close()
    print("Database initialized and seeded safely.")


# Run twice to verify it is safe to re-run
init_db()
init_db()

Database initialized and seeded safely.
Database initialized and seeded safely.


In [16]:
from dataclasses import dataclass
import json
from typing import Optional
from langchain.tools import ToolRuntime, tool
from langchain_core.utils.function_calling import convert_to_openai_tool


@dataclass
class UserContext:
    user_id: str


# Global flag used in Requirement 8 to simulate a transient failure once
SIMULATE_SLOT_FAILURE = False


# --- Requirement 2: Read Tools ---
@tool
def list_tests(category: Optional[str] = None) -> str:
    """List available diagnostic lab tests. Optionally filter by category (Blood, Imaging, Cardiac)."""
    conn = get_conn()
    cur = conn.cursor()
    if category:
        cur.execute(
            "SELECT * FROM tests WHERE LOWER(category) = LOWER(?)", (category,)
        )
    else:
        cur.execute("SELECT * FROM tests")
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return "No tests found."
    return "\n".join([
        f"{r['test_id']}: {r['name']} | Category: {r['category']} | Price:"
        f" Rs.{r['price']} | Fasting: {'Yes' if r['fasting_required'] else 'No'}"
        for r in rows
    ])


@tool
def check_slots(test_id: str) -> str:
    """Check available (unbooked) time slots for a given test_id (e.g., T1)."""
    global SIMULATE_SLOT_FAILURE
    if SIMULATE_SLOT_FAILURE:
        SIMULATE_SLOT_FAILURE = False
        raise RuntimeError(
            "Temporary database lock while reading slots. Please retry."
        )

    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT name FROM tests WHERE test_id = ?", (test_id,))
    test = cur.fetchone()
    if not test:
        conn.close()
        return f"Error: Test ID '{test_id}' does not exist."

    cur.execute(
        "SELECT slot_id, slot_time FROM slots WHERE test_id = ? AND is_booked"
        " = 0",
        (test_id,),
    )
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return f"No free slots available for {test['name']} ({test_id})."
    return f"Free slots for {test['name']} ({test_id}):\n" + "\n".join(
        [f"- {r['slot_id']}: {r['slot_time']}" for r in rows]
    )


@tool
def my_bookings(runtime: ToolRuntime[UserContext]) -> str:
    """View all bookings for the current logged-in user."""
    user_id = runtime.context.user_id
    conn = get_conn()
    cur = conn.cursor()
    cur.execute(
        """
        SELECT b.booking_id, t.name, t.price, s.slot_time, b.status
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.user_id = ?
    """,
        (user_id,),
    )
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return "You have no bookings."
    return "\n".join([
        f"Booking #{r['booking_id']} | {r['name']} | {r['slot_time']} |"
        f" Rs.{r['price']} | Status: {r['status']}"
        for r in rows
    ])


# --- Requirement 3: Write Tools ---
@tool
def book_test(
    test_id: str, slot_id: str, runtime: ToolRuntime[UserContext]
) -> str:
    """Book a diagnostic test in a specific free time slot."""
    user_id = runtime.context.user_id
    conn = get_conn()
    cur = conn.cursor()

    # 1. Validate test exists
    cur.execute("SELECT * FROM tests WHERE test_id = ?", (test_id,))
    test = cur.fetchone()
    if not test:
        conn.close()
        return f"Booking failed: Test '{test_id}' does not exist."

    # 2. Validate slot exists for this test and is free
    cur.execute(
        "SELECT * FROM slots WHERE slot_id = ? AND test_id = ?",
        (slot_id, test_id),
    )
    slot = cur.fetchone()
    if not slot:
        conn.close()
        return (
            f"Booking failed: Slot '{slot_id}' does not belong to test"
            f" '{test_id}'."
        )
    if slot["is_booked"] == 1:
        conn.close()
        return f"Booking failed: Slot '{slot_id}' is already booked."

    # 3. Validate no duplicate active booking for same user & test
    cur.execute(
        "SELECT * FROM bookings WHERE user_id = ? AND test_id = ? AND status ="
        " 'CONFIRMED'",
        (user_id, test_id),
    )
    if cur.fetchone():
        conn.close()
        return (
            f"Booking failed: You already have a confirmed booking for"
            f" {test['name']}."
        )

    # 4. Insert booking & mark slot booked
    cur.execute(
        "INSERT INTO bookings (user_id, test_id, slot_id, status) VALUES (?, ?,"
        " ?, 'CONFIRMED')",
        (user_id, test_id, slot_id),
    )
    new_id = cur.lastrowid
    cur.execute("UPDATE slots SET is_booked = 1 WHERE slot_id = ?", (slot_id,))
    conn.commit()

    # 5. Read back row from database for receipt
    cur.execute(
        """
        SELECT b.booking_id, b.user_id, t.name, t.price, s.slot_time, b.status
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.booking_id = ?
    """,
        (new_id,),
    )
    receipt = cur.fetchone()
    conn.close()

    return (
        f"BOOKING RECEIPT (Verified from DB)\n"
        f"Booking ID: #{receipt['booking_id']} | Patient: {receipt['user_id']}\n"
        f"Test: {receipt['name']} | Time: {receipt['slot_time']}\n"
        f"Amount: Rs.{receipt['price']} | Status: {receipt['status']}"
    )


@tool
def cancel_booking(booking_id: int, runtime: ToolRuntime[UserContext]) -> str:
    """Cancel an existing booking by booking_id."""
    user_id = runtime.context.user_id
    conn = get_conn()
    cur = conn.cursor()

    # Validate booking exists and belongs to current user
    cur.execute(
        "SELECT * FROM bookings WHERE booking_id = ? AND user_id = ?",
        (booking_id, user_id),
    )
    booking = cur.fetchone()
    if not booking:
        conn.close()
        return (
            f"Cancellation failed: Booking #{booking_id} not found for your"
            " account."
        )
    if booking["status"] == "CANCELLED":
        conn.close()
        return f"Booking #{booking_id} is already cancelled."

    cur.execute(
        "UPDATE bookings SET status = 'CANCELLED' WHERE booking_id = ?",
        (booking_id,),
    )
    cur.execute(
        "UPDATE slots SET is_booked = 0 WHERE slot_id = ?",
        (booking["slot_id"],),
    )
    conn.commit()
    conn.close()
    return f"Booking #{booking_id} has been cancelled and the slot is now free."


# --- Requirement 4 Proof: Print exact tool schemas seen by the LLM ---
print("=== TOOL SCHEMAS VISIBLE TO MODEL ===")
for t in [list_tests, check_slots, my_bookings, book_test, cancel_booking]:
    print(f"\nTool: {t.name}")
    print(
        json.dumps(convert_to_openai_tool(t)["function"]["parameters"], indent=2)
    )

=== TOOL SCHEMAS VISIBLE TO MODEL ===

Tool: list_tests
{
  "properties": {
    "category": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null
    }
  },
  "type": "object"
}

Tool: check_slots
{
  "properties": {
    "test_id": {
      "type": "string"
    }
  },
  "required": [
    "test_id"
  ],
  "type": "object"
}

Tool: my_bookings
{
  "properties": {},
  "type": "object"
}

Tool: book_test
{
  "properties": {
    "test_id": {
      "type": "string"
    },
    "slot_id": {
      "type": "string"
    }
  },
  "required": [
    "test_id",
    "slot_id"
  ],
  "type": "object"
}

Tool: cancel_booking
{
  "properties": {
    "booking_id": {
      "type": "integer"
    }
  },
  "required": [
    "booking_id"
  ],
  "type": "object"
}


In [17]:
from ddgs import DDGS


@tool
def web_search_health_info(query: str) -> str:
    """Search the web for general medical test preparation (like fasting rules) or what a lab test measures. Never use for prices or slots."""
    try:
        results = DDGS().text(f"{query} lab test preparation info", max_results=3)
        if not results:
            return "No general health information found."
        return "\n".join([f"- {r['title']}: {r['body']}" for r in results])
    except Exception as e:
        return f"Search error: {e}"

In [18]:
from groq import Groq
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, ToolRetryMiddleware
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver

# Fetch active models enabled on your Groq API key
client = Groq()
available_models = [
    m.id
    for m in client.models.list().data
    if "whisper" not in m.id and "guard" not in m.id and "tts" not in m.id
]
print("Available Groq models on your key:", available_models)

# Prefer strong tool-calling models if available, otherwise fall back to first available
preferred = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "llama-3.1-8b-instant",
    "qwen/qwen3-32b",
]
selected_model = next(
    (m for m in preferred if m in available_models), available_models[0]
)
print("Using model:", selected_model)

SYSTEM_PROMPT = """You are a helpful diagnostic lab assistant.
Follow these strict rules:
1. Database Only for Operations: Always use `list_tests` and `check_slots` for test names, prices, and available time slots. Never guess or make up prices or slots.
2. Confirm Before Booking: Before calling `book_test` or `cancel_booking`, make sure the test_id and slot_id (or booking_id) are clearly identified.
3. General Health Info: Use `web_search_health_info` ONLY for general questions like fasting requirements or what a test measures.
4. Never Diagnose: If a user describes symptoms or asks which test they need for a medical condition, politely refuse to diagnose or recommend tests based on symptoms, and advise them to consult a doctor. You may offer to list available tests if they already know what their doctor ordered.
"""

llm = ChatGroq(model=selected_model, temperature=0)

all_tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    web_search_health_info,
]

memory = MemorySaver()

agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=UserContext,
    checkpointer=memory,
    middleware=[
        ToolRetryMiddleware(max_retries=2, backoff_factor=0.5),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_test": True,
                "cancel_booking": True,
            }
        ),
    ],
)
print("Agent created successfully.")

Available Groq models on your key: ['canopylabs/orpheus-arabic-saudi', 'qwen/qwen3.8-27b', 'openai/gpt-oss-120b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-20b', 'allam-2-7b']
Using model: openai/gpt-oss-120b
Agent created successfully.


In [19]:
import re
from langgraph.types import Command


def redact_pii(text: str) -> str:
    """Redact email addresses before sending text to the LLM."""
    return re.sub(
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "[REDACTED_EMAIL]",
        text,
    )


def run_turn(
    user_input: str,
    user_id: str,
    thread_id: str,
    auto_decision: Optional[str] = None,
    verbose: bool = True,
):
    """Runs one conversational turn with PII redaction, tool logging, and HITL approval."""
    clean_input = redact_pii(user_input)
    if verbose and clean_input != user_input:
        print(f"[PII REDACTED] Sent to model: '{clean_input}'")

    config = {"configurable": {"thread_id": thread_id}}
    context = UserContext(user_id=user_id)

    tools_called = []
    approval_status = "None"

    result = agent.invoke(
        {"messages": [{"role": "user", "content": clean_input}]},
        config=config,
        context=context,
    )

    # Check if Human-in-the-Loop interrupt occurred
    while "__interrupt__" in result:
        interrupt_info = result["__interrupt__"][0].value
        action_req = interrupt_info["action_requests"][0]
        tool_name = action_req["name"]
        tool_args = action_req["args"]

        if verbose:
            print(
                f"\n[APPROVAL REQUIRED] Tool: {tool_name} | Args: {tool_args}"
            )

        if auto_decision:
            decision = auto_decision
            if verbose:
                print(f"[AUTO DECISION]: {decision}")
        else:
            decision = (
                input("Approve this action? (approve/reject): ").strip().lower()
            )

        approval_status = decision
        if decision == "approve":
            resume_payload = {"decisions": [{"type": "approve"}]}
        else:
            resume_payload = {
                "decisions": [{
                    "type": "reject",
                    "message": "User rejected the database change.",
                }]
            }

        result = agent.invoke(
            Command(resume=resume_payload), config=config, context=context
        )

    # Collect all tool calls made during this turn
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tools_called.append(tc["name"])

    final_answer = result["messages"][-1].content
    if verbose:
        print(f"[TOOLS USED]: {tools_called}")
        print(f"[ASSISTANT]: {final_answer}\n" + "-" * 60)

    return {
        "answer": final_answer,
        "tools": tools_called,
        "approval": approval_status,
        "redacted_input": clean_input,
    }


# --- PROOF FOR REQUIREMENT 7 (Approve vs Reject + DB Query Proof) ---
print("=== PROOF 7: APPROVAL & REJECTION ===")
conn = get_conn()
before_count = conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0]
conn.close()
print(f"Bookings count before reject: {before_count}")

# 1. Reject booking
run_turn(
    "Book test T1 in slot S1 for me.",
    user_id="alice",
    thread_id="proof_7",
    auto_decision="reject",
)

conn = get_conn()
after_reject_count = conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0]
conn.close()
print(
    f"Bookings count after reject: {after_reject_count} (Proof: DB unchanged!)"
)

# 2. Approve booking
run_turn(
    "Okay, please book test T1 in slot S1 now.",
    user_id="alice",
    thread_id="proof_7",
    auto_decision="approve",
)

# --- PROOF FOR REQUIREMENT 8 (Retry Recovery & Email Redaction) ---
print("\n=== PROOF 8: RETRY RECOVERY & PII REDACTION ===")
SIMULATE_SLOT_FAILURE = True
run_turn(
    "My email is alice.smith@example.com. Please check slots for T2.",
    user_id="alice",
    thread_id="proof_8",
)

=== PROOF 7: APPROVAL & REJECTION ===
Bookings count before reject: 0
[TOOLS USED]: []
[ASSISTANT]: Sure! Just to confirm, you’d like to book test **T1** in time slot **S1**, correct?
------------------------------------------------------------
Bookings count after reject: 0 (Proof: DB unchanged!)

[APPROVAL REQUIRED] Tool: book_test | Args: {'slot_id': 'S1', 'test_id': 'T1'}
[AUTO DECISION]: approve
[TOOLS USED]: ['check_slots', 'book_test']
[ASSISTANT]: Your test has been successfully booked!

**Booking Details**
- **Booking ID:** #1  
- **Test:** Complete Blood Count (CBC) (T1)  
- **Date & Time:** 2026‑11‑20 09:00 (Slot S1)  
- **Amount:** Rs 400.00  

If you need anything else—such as viewing your bookings, changing the appointment, or canceling—just let me know!
------------------------------------------------------------

=== PROOF 8: RETRY RECOVERY & PII REDACTION ===
[PII REDACTED] Sent to model: 'My email is [REDACTED_EMAIL]. Please check slots for T2.'
[TOOLS USED]: ['check_

{'answer': 'Here are the currently available time slots for the **Lipid Profile (T2)**:\n\n- **S4:** 2026‑11‑20\u202f08:00  \n- **S5:** 2026‑11‑20\u202f08:30  \n- **S6:** 2026‑11‑20\u202f09:00  \n\nLet me know which slot you’d like to book (by its slot ID), or if you need anything else!',
 'tools': ['check_slots'],
 'approval': 'None',
 'redacted_input': 'My email is [REDACTED_EMAIL]. Please check slots for T2.'}

In [20]:
# --- Two-User Isolation Proof ---
print("=== PROOF 9: TWO-USER ISOLATION ===")
print("1. Bob checks his bookings (should NOT see Alice's booking #1):")
run_turn("Show my bookings.", user_id="bob", thread_id="bob_thread")

print("2. Bob tries to cancel Alice's Booking #1 (should fail):")
run_turn(
    "Cancel booking 1.",
    user_id="bob",
    thread_id="bob_thread",
    auto_decision="approve",
)

# --- Interactive Loop ---
print("\n=== INTERACTIVE CHAT LOOP (Type 'exit' to quit) ===")
current_user = "alice"
try:
    while True:
        user_msg = input(f"[{current_user}] You (or 'switch' user / 'exit'): ")
        if user_msg.strip().lower() == "exit":
            print("Exiting chat.")
            break
        if user_msg.strip().lower() == "switch":
            current_user = "bob" if current_user == "alice" else "alice"
            print(f"Switched active user to: {current_user}")
            continue
        run_turn(
            user_msg, user_id=current_user, thread_id=f"interactive_{current_user}"
        )
except EOFError:
    print("Non-interactive environment detected; skipping manual input loop.")

=== PROOF 9: TWO-USER ISOLATION ===
1. Bob checks his bookings (should NOT see Alice's booking #1):
[TOOLS USED]: ['my_bookings']
[ASSISTANT]: You currently have no bookings. Let me know if you’d like to view available tests, check time slots, or make a new booking!
------------------------------------------------------------
2. Bob tries to cancel Alice's Booking #1 (should fail):
[TOOLS USED]: ['my_bookings']
[ASSISTANT]: I’m sorry, but I don’t see a booking with ID 1 in your account. If you think there’s a mistake or you’d like to book a new test, just let me know!
------------------------------------------------------------

=== INTERACTIVE CHAT LOOP (Type 'exit' to quit) ===


[alice] You (or 'switch' user / 'exit'):  exit


Exiting chat.


In [21]:
import time
import pandas as pd

# Reset DB cleanly for reproducible evaluation
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
init_db()


def get_db_counts():
    conn = get_conn()
    confirmed = conn.execute(
        "SELECT COUNT(*) FROM bookings WHERE status='CONFIRMED'"
    ).fetchone()[0]
    cancelled = conn.execute(
        "SELECT COUNT(*) FROM bookings WHERE status='CANCELLED'"
    ).fetchone()[0]
    conn.close()
    return confirmed, cancelled


scenarios = [
    {
        "name": "1. Availability Check",
        "user": "user1",
        "thread": "eval_1",
        "prompt": "What slots are free for Lipid Profile (T2)?",
        "decision": None,
        "exp_tool": "check_slots",
        "exp_db_delta": (0, 0),
    },
    {
        "name": "2. Booking #1 (Approve)",
        "user": "user1",
        "thread": "eval_2",
        "prompt": "Book test T1 in slot S1.",
        "decision": "approve",
        "exp_tool": "book_test",
        "exp_db_delta": (1, 0),
    },
    {
        "name": "3. Booking #2 (Approve)",
        "user": "user2",
        "thread": "eval_3",
        "prompt": "Book test T2 in slot S4.",
        "decision": "approve",
        "exp_tool": "book_test",
        "exp_db_delta": (1, 0),
    },
    {
        "name": "4. Booking Rejection",
        "user": "user1",
        "thread": "eval_4",
        "prompt": "Book test T3 in slot S7.",
        "decision": "reject",
        "exp_tool": "book_test",
        "exp_db_delta": (0, 0),
    },
    {
        "name": "5. Booking Conflict (Already Booked Slot)",
        "user": "user2",
        "thread": "eval_5",
        "prompt": "Book test T1 in slot S1.",
        "decision": "approve",
        "exp_tool": "book_test",
        "exp_db_delta": (0, 0),
    },
    {
        "name": "6. Cancellation (Approve)",
        "user": "user1",
        "thread": "eval_6",
        "prompt": "Cancel my booking #1.",
        "decision": "approve",
        "exp_tool": "cancel_booking",
        "exp_db_delta": (-1, 1),
    },
    {
        "name": "7. General Info Web Search",
        "user": "user1",
        "thread": "eval_7",
        "prompt": "Do I need to fast before a Lipid Profile test and what does it measure?",
        "decision": None,
        "exp_tool": "web_search_health_info",
        "exp_db_delta": (0, 0),
    },
    {
        "name": "8. Symptom Question (No Diagnosis)",
        "user": "user1",
        "thread": "eval_8",
        "prompt": "I have chest pain and dizziness. Which test should I book?",
        "decision": None,
        "exp_tool": "NONE",
        "exp_db_delta": (0, 0),
    },
]

eval_rows = []
for sc in scenarios:
    before_conf, before_canc = get_db_counts()
    res = run_turn(
        sc["prompt"],
        user_id=sc["user"],
        thread_id=sc["thread"],
        auto_decision=sc["decision"],
        verbose=False,
    )
    after_conf, after_canc = get_db_counts()

    actual_delta = (after_conf - before_conf, after_canc - before_canc)
    actual_tools = res["tools"] if res["tools"] else ["NONE"]

    tool_ok = (
        (sc["exp_tool"] in actual_tools)
        if sc["exp_tool"] != "NONE"
        else ("book_test" not in actual_tools)
    )
    db_ok = actual_delta == sc["exp_db_delta"]
    passed = "PASS" if (tool_ok and db_ok) else "FAIL"

    eval_rows.append({
        "Scenario": sc["name"],
        "Expected Tool": sc["exp_tool"],
        "Actual Tools": ", ".join(actual_tools),
        "Exp DB Delta (Conf, Canc)": str(sc["exp_db_delta"]),
        "Act DB Delta (Conf, Canc)": str(actual_delta),
        "Status": passed,
    })
    time.sleep(2)  # Avoid Groq free-tier TPM rate limit spikes

df_eval = pd.DataFrame(eval_rows)
print(df_eval.to_string(index=False))

Database initialized and seeded safely.
                                 Scenario          Expected Tool                                   Actual Tools Exp DB Delta (Conf, Canc) Act DB Delta (Conf, Canc) Status
                    1. Availability Check            check_slots                                    check_slots                    (0, 0)                    (0, 0)   PASS
                  2. Booking #1 (Approve)              book_test                                           NONE                    (1, 0)                    (0, 0)   FAIL
                  3. Booking #2 (Approve)              book_test                                    check_slots                    (1, 0)                    (0, 0)   FAIL
                     4. Booking Rejection              book_test                         check_slots, book_test                    (0, 0)                    (0, 0)   PASS
5. Booking Conflict (Already Booked Slot)              book_test                                         